In [ ]:
class Reserve:
    def __init__(self):
        self._cartes = {
            # Etablissements bleus
            "Champ de blé": {
                "objet": champ_ble,
                "quantite": 6
            },
            "Ferme": {
                "objet": ferme,
                "quantite": 6
            },
            "Forêt": {
                "objet": foret,
                "quantite": 6
            },
            "Verger": {
                "objet": verger,
                "quantite": 6
            },
            "Mine": {
                "objet": mine,
                "quantite": 6
            },

            # Etablissements verts
            "Boulangerie": {
                "objet": boulangerie,
                "quantite": 6
            },
            "Supérette": {
                "objet": superette,
                "quantite": 6
            },
            "Fromagerie": {
                "objet": fromagerie,
                "quantite": 6
            },
            "Fabrique de meubles": {
                "objet": fabrique,
                "quantite": 6
            },
            "Marché de fruits et légumes": {
                "objet": marché,
                "quantite": 6
            },

            # Etablissements rouges
            "Café": {
                "objet": cafe,
                "quantite": 6
            },
            "Restaurant": {
                "objet": resto,
                "quantite": 6
            },

            # Etablissements violets
            # j'ai créé les instances avec Stade(), Centre_daff() et Chaine()
            # il faudra rajouter les 3 instances violettes
            "Stade": {
                "objet": stade,
                "quantite": 4
            },
            "Centre d'affaires": {
                "objet": centre_daff,
                "quantite": 4
            },
            "Chaîne de télévision": {
                "objet": chaine,
                "quantite": 4
            }
        }
    def est_disponible(self, nom:str)->bool:
        if nom in self._cartes and self._cartes[nom]["quantite"] > 0: 
            return True
        return False
        
    def prendre_carte(self, nom: str)-> Etablissement | None:
        if self.est_disponible(nom):
            self._cartes[nom]["quantite"] -= 1
            return self._cartes[nom]["objet"] # retire un exemplaire du stock et retourne l'objet correspondant
                                                # afin qu'il puisse être ajouté à la ville du joueur
        return None

class Monument:
    def __init__(self, nom: str, cout: int)->None:
        self._nom = nom
        self._cout = cout
        self._construit = False #car le joueur ne possede pas encore ce monument
    
    @property
    def nom(self)->str:
        return self._nom

    @property
    def cout(self)->int:
        return self._cout

    @property
    def construit(self)->bool:
        return self._construit

    def construire(self)->None: #pour changer son etat
        self._construit = True


class Ville:
    def __init__(self)->None:
        self._etablissements = [champ_ble, boulangerie]
        self._monuments = [
            Monument("Gare", 4),
            Monument("Centre commercial", 10),
            Monument("Parc d'attractions", 16),
            Monument("Tour radio", 22)
        ]

    @property
    def monuments(self)->list[Monument]:
        return self._monuments

    def ajouter_carte(self, carte: Etablissement)->None:
    self._etablissements.append(carte)

    def echanger_carte(self):
        pass

class Partie:
    def __init__(self, joueurs:list["Joueur"]):
        self._joueurs=joueurs
        self._reserve = Reserve()
        self._joueur_actuel = 0
#prendre en compte : 1 tour par joueur avant que qqn ait "Gare", paiement avant revenu, 

    def lancer_de(self): #sens horaire
        pass

    def tour_joueur(self, resultat):
        resultat=self.lancer_de()
        self.activer_carte(resultat)
        self.achat_cartes()

    def joueur_suivant(self):
        pass
    
    def activer_carte(self, resultat):
        pass

    def paiement(self):
        pass

    def multiples_paiements(self): #sens antihoraire
        pass

    def revenu(self):
        pass

    def achat_cartes(self,nom: str)-> bool:
    # au lieu de retourner la carte comme dans le jeu, ce que je propose c'est qu'on procède comme un achat
    # pas besoin de verifier la quantité des monuments pour les achats!!
        joueur = self._joueurs[self._joueur_actuel] 

        #cas1 :Etablissemnt
        if nom in self._reserve._cartes: # verifie si c'est un établissement
            if not self._reserve.est_disponible(nom):
                print("Cette carte n'est plus disponible.")
                return False     
            
            carte = self._reserve._cartes[nom]["objet"] # ici j'ai recupere l'objet 
                                                       #pour pouvoir acceder au cout de la carte
            prix = carte.cout

            if joueur.argent >= prix:
                joueur.payer(prix)
                carte = self._reserve.prendre_carte(nom)
                joueur.ville.ajouter_carte(carte) 
                return True 
            
            print("Le joueur n'a pas assez d'argent")
            return False

        # Cas 2: monument
        else:
            for monument in joueur.ville.monuments:

                if monument.nom == nom:

                    if monument.construit:
                        print("Monument déjà construit")
                        return False

                    if joueur.argent >= monument.cout:
                        joueur.payer(monument.cout)
                        monument.construire()
                        return True

                    print("Pas assez d'argent")
                    return False

            print("Ce monument n'existe pas.")
            return False

        

 
    def choisir_joueur(self)->"Joueur": #petite correction pour le nom vu que ce nom etait deja defini dans la classe Chaine
        proprietaire = self._joueurs[self._joueur_actuel]
        adversaires = []

        for joueur in self._joueurs:
            if joueur != proprietaire:
                adversaires.append(joueur) #on recupere la liste des adversaires

        for i in range(len(adversaires)):
            print(i + 1, adversaires[i].nom)#on affiche les adversaires numerotes avec leurs noms, i+1 juste pour commencer a compter a partir de 1
        
        choix = input("Choisissez un adversaire : ")
        while not choix.isdigit() or int(choix)<1 or int(choix)>len(adversaires):#.isdigit permet de verifier entier>0
            choix = input("Choix invalide. Réessayez : ")

        return adversaires[int(choix) - 1] #-1 car les indices commencent a 0 et int() car choix est str


    def victoire(self)->bool:
        joueur = self._joueurs[self._joueur_actuel]
        monuments = joueur.ville.monuments
        for monument in monuments:
            if monument.construit == False: #si on trouve un monument non construit False donc pas gagne
                return False

        return True #si tous les monuments sont construits